In [20]:
import os, sys
from datetime import date, timedelta
from pyspark.sql import SparkSession, functions

In [2]:
# settings
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [3]:
spark = (
    SparkSession.builder
    .appName("df-load")
    .master("local[1]")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

# 00 - Criando um DataFrame

In [4]:
data = {
    "id": list(range(1, 11)),
    "value": [10.5, 20.1, 30.2, 40.8, 50.0, 60.3, 70.7, 80.9, 90.4, 100.6],
    "name": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"],
    "active": [True, False, True, True, False, True, False, True, False, True],
    "date": [date(2024, 1, 1) + timedelta(days=i) for i in range(10)]
}

In [5]:
rows = list(zip(
    data["id"],
    data["value"],
    data["name"],
    data["active"],
    data["date"],
))

In [11]:
columns = ["id", "value", "name", "active", "date"]

df = spark.createDataFrame(rows, columns)

In [12]:
# tipos das colunas
df

DataFrame[id: bigint, value: double, name: string, active: boolean, date: date]

In [13]:
df.show()

+---+-----+----+------+----------+
| id|value|name|active|      date|
+---+-----+----+------+----------+
|  1| 10.5|   A|  true|2024-01-01|
|  2| 20.1|   B| false|2024-01-02|
|  3| 30.2|   C|  true|2024-01-03|
|  4| 40.8|   D|  true|2024-01-04|
|  5| 50.0|   E| false|2024-01-05|
|  6| 60.3|   F|  true|2024-01-06|
|  7| 70.7|   G| false|2024-01-07|
|  8| 80.9|   H|  true|2024-01-08|
|  9| 90.4|   I| false|2024-01-09|
| 10|100.6|   J|  true|2024-01-10|
+---+-----+----+------+----------+



In [14]:
df.createOrReplaceTempView("table")

In [16]:
# filtros simples
# usando queries
spark.sql("""
    SELECT *
    FROM table
    WHERE active = true
""").show()

+---+-----+----+------+----------+
| id|value|name|active|      date|
+---+-----+----+------+----------+
|  1| 10.5|   A|  true|2024-01-01|
|  3| 30.2|   C|  true|2024-01-03|
|  4| 40.8|   D|  true|2024-01-04|
|  6| 60.3|   F|  true|2024-01-06|
|  8| 80.9|   H|  true|2024-01-08|
| 10|100.6|   J|  true|2024-01-10|
+---+-----+----+------+----------+



In [19]:
# spark é lazy por isso usamos show()
df.filter(df.active == True).show()

+---+-----+----+------+----------+
| id|value|name|active|      date|
+---+-----+----+------+----------+
|  1| 10.5|   A|  true|2024-01-01|
|  3| 30.2|   C|  true|2024-01-03|
|  4| 40.8|   D|  true|2024-01-04|
|  6| 60.3|   F|  true|2024-01-06|
|  8| 80.9|   H|  true|2024-01-08|
| 10|100.6|   J|  true|2024-01-10|
+---+-----+----+------+----------+



In [22]:
df.filter(functions.col('active') == True).show()

+---+-----+----+------+----------+
| id|value|name|active|      date|
+---+-----+----+------+----------+
|  1| 10.5|   A|  true|2024-01-01|
|  3| 30.2|   C|  true|2024-01-03|
|  4| 40.8|   D|  true|2024-01-04|
|  6| 60.3|   F|  true|2024-01-06|
|  8| 80.9|   H|  true|2024-01-08|
| 10|100.6|   J|  true|2024-01-10|
+---+-----+----+------+----------+



In [23]:
df.filter('active == True').show()

+---+-----+----+------+----------+
| id|value|name|active|      date|
+---+-----+----+------+----------+
|  1| 10.5|   A|  true|2024-01-01|
|  3| 30.2|   C|  true|2024-01-03|
|  4| 40.8|   D|  true|2024-01-04|
|  6| 60.3|   F|  true|2024-01-06|
|  8| 80.9|   H|  true|2024-01-08|
| 10|100.6|   J|  true|2024-01-10|
+---+-----+----+------+----------+



In [25]:
df.filter(df.value > 50).show()

+---+-----+----+------+----------+
| id|value|name|active|      date|
+---+-----+----+------+----------+
|  6| 60.3|   F|  true|2024-01-06|
|  7| 70.7|   G| false|2024-01-07|
|  8| 80.9|   H|  true|2024-01-08|
|  9| 90.4|   I| false|2024-01-09|
| 10|100.6|   J|  true|2024-01-10|
+---+-----+----+------+----------+



# 01 - Lendo um Arquivo

In [26]:
spark = SparkSession.builder.appName("ExemploLeitura").getOrCreate()

In [28]:
df = spark.read.csv(
    "data/data_raw.csv", 
    header=True,
    inferSchema=True,
    sep=","
)

In [30]:
df.show(5)

+-----+-----------+------------+-------------+----------+------------------+-------+---------+
|sales|customer_id|customer_age|purchase_date|   product|last_purchase_date|  price|seller_id|
+-----+-----------+------------+-------------+----------+------------------+-------+---------+
|   52|       1511|          86|   2024-10-07|    Laptop|        2022-05-29|2534.07|      199|
|   93|       4985|          52|   2023-12-10|Smartphone|        2023-09-20|1879.78|      182|
|   15|       3092|          69|   2024-01-06|        TV|        2022-07-06|4844.32|      112|
|   72|       4179|          31|   2023-02-02|Smartphone|        2022-01-20|4705.96|      147|
|   61|       3266|          80|   2024-01-09|    Tablet|        2023-01-08|4766.52|      159|
+-----+-----------+------------+-------------+----------+------------------+-------+---------+
only showing top 5 rows

